# 5. Photometry - Under construction!

In this we give a thorough introduction to extracting and visualising the light curves produced by the in-build photometry algorithm. The algorithm implemented into PlatoSim is Marchiori et al. (2019)'s optimal binary mask aperture routine which will be used for the photometric flux extraction of the P5 sample on-board the spacecraft.

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# To interact with the plot use
%matplotlib notebook

### Imports

In [ ]:
import os
import h5py
import random
import numpy as np
import matplotlib.pyplot as plt
from prettytable import PrettyTable

# PlatoSim
import platosim.plot            as pt
import platosim.referenceFrames as rf
from platosim.photometry   import Photometry
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

---
## 5.1 - Implemented photometry algorithms
---

We assume that each pixel value $f_{ij}$ is composed of a signal $s_{ij}$, a background $b_{ij}$, and noise $\epsilon_{ij}$:  

$$ f_{ij} = s_{ij} + b_{ij} + \varepsilon_{ij}$$

The background $b_{ij}$ is assumed to be known (in practice: read from the HDF5 file), and subtracted first.

What did the photometry script do?
- Estimate the bias and subtract it
- Correct for open shutter smearing using the smearing maps
- Convert from [ADU] to [electrons] using the (FEE and CCD) gain
- Correct for the flatfield
- Subtract the background (assumed to be known)
- Do the weighted mask photometry, assuming the correct star positions

### 5.1.1 - Optimal Aperture Photometry (OAP)

The OAP method is the currenly planned photometry technique to use for stars processed on-board the spacecraft (due to limited amount of telemetry capabilities). A full description can by found in [Marchiori+2019] paper and in essence it is optimised to provide the lowest Noise-to-Signal Ratio (NSR) needed for detecting the transit signature of exoplanets. However, we note that this method not necessarily provides the optimal description for stellar pulsation and variability (seen in the Fourier domain), but for which ultimately will be a trade-off between including a much of the stellar signal while still trying to avoid stellar contamination. 

### 5.1.2 - Weighted OAP photometry

The total flux of a star is then computed using a weighted sum:

$$ F \equiv \sum\limits_{ij} \alpha \ w_{ij} (f_{ij} - b_{ij}) $$

where the weights are exponentially decaying:

$$w_{ij} = \exp\left(-\frac{(i-i_0)^2 + (j-j_0)^2}{2\sigma^2_{\rm PSF}}\right)$$

The exact (fractional) pixel coordinates $(i_0, j_0)$ of the star will be known using the Gaia catalog. For now, they are assumed to be known (i.e. read from the HDF5 file) as well.

#### Figure out the width of the PSF

In [ ]:
# stop
outputFile = h5py.File("photometrySimulation.hdf5")

In [ ]:
psf = array(outputFile["/PSF/rebinnedPSFsubPixel"])
NsubPixels = outputFile["/InputParameters/SubField"].attrs["SubPixels"]
Npixels = int(psf.shape[0]/NsubPixels)
print("PSF consists of {0}x{0} pixels, with each pixel made up of {1}x{1} subpixels".format(Npixels, NsubPixels))

Plot the PSF to see what we're dealing with.

In [ ]:
plt.pcolormesh(psf, cmap=cm.nipy_spectral, vmax=0.01)                         # 'vmax' to get a prettier image
plt.grid(True, which='major', axis='both', linestyle='-', color='w')     # To get a 'pixel-grid' on top of it.
plt.xticks(arange(0, psf.shape[0], NsubPixels))
plt.yticks(arange(0, psf.shape[1], NsubPixels))
plt.xlim(0, psf.shape[0])
plt.ylim(0, psf.shape[1])

In [ ]:
sigmaPSF = phot.computePSFsigma(psf, NsubPixels, 500)
print("Standard deviation of the PSF (assuming symmetry): {0}".format(sigmaPSF))

The following function extracts the flux of all stars in every image in the HDF5 simulation output file, and writes the results to a second HDF5 file. Depending on the number of exposures, this can take a while.

In [ ]:
print(outputFile["Flatfield"])

In [ ]:
reload(phot)
print("Start")
phot.photometry("photometrySimulation.hdf5", "photometryOutput.hdf5",starIDsInImage0, maxNexposures=1000)
print("Done")

In [ ]:
%ls -lh photometryOutput.hdf5

---
## 5.2 - Include photometry in your simulations
---

As an initial minimal example we just simulate a single exposure:

In [ ]:
# Set up a Simulation object
outputFileName = "output_example1"
sim = Simulation(outputFileName, outputDir=os.getcwd())
sim["ObservingParameters/NumExposures"] = 1

In [ ]:
# Run the simulation
simfile = sim.run(removeOutputFile=True)

And as usually we can plot the simulated images with:

In [ ]:
fig, ax = simfile.showImage(imageNr=False, clipPercentile=1, imgScale="clip", 
                            showStarPositions=True, colorMap="magma", 
                            colorBar=True, showGrid=True) 

We can make a little animation of how the mask update strategy looks like:

In [ ]:
from platosim.plot import plotSubfieldAnimation

In [ ]:
idir = "/lhome/nicholas/software/workdir/test/output"
filename = idir + "/000000000_Ncam1.1_Q1.hdf5"
plotSubfieldAnimation(filename, outputFileName=idir+"/animation", showStarPositions="PIC", 
                      skipNimages=1000, showMaskOfStarID=1)

### 5.1.1 - Photometry of all stars in a larger subfield

First we fetch the stellar IDs:

In [ ]:
starID, row, col, Xmm, Ymm, flux = simfile.getStarCoordinates(0)
starID

With the stellar IDs we can now create a photometry list necessity for PlatoSim to perform the photometric extraction:

In [ ]:
photometryFile = os.getcwd() + "/photometry.txt"
sim.createPhotometryTargetFile(starID, photometryFile)

As a minimal example we here simulate 100 images:

In [ ]:
sim["ObservingParameters/NumExposures"] = 100
sim["Photometry/IncludePhotometry"]     = True
sim["PSF/Model"]                        = "AnalyticNonGaussian"

In [ ]:
# Run the simulation
simfile = sim.run(removeOutputFile=True)

### 5.1.2 - Photometry of a single target in a imagette

Since we now only simulate an imagette we can simulate a substaintially longer time series. Lets simulate a full quarter of a year, i.e. 90 days, being the longest continuous duration any time series as the spacecraft needs to realign its solar panels towards the Sun every quarter. 

---
## 5.2 - Extract photometry from HDF5
---

To seperate the post-processing for pixel maps and the photometry, a dedicated class called `Photometry()` has been developed to easily handle the photometric data output of PlatoSim's HDF5 output files. First we fetch the HDF5 output file:

In [ ]:
f = Photometry(outputFileName + ".hdf5")

The `Photometry()` class has multiple utilities to extract information. You can investigate this class with e.g. `PrettyTable`. We will show a few of the functionalities in the following:

In [ ]:
t = PrettyTable()
t.add_column("Photometry functions", dir(Photometry)[27:])
t

`getTime()`: Get the time column simulated:

In [ ]:
f.getTime().head()

`getFlux()`: Get the flux column for either a specific star or all stars simulated:

In [ ]:
f.getFlux(starID[0]).head()

In [ ]:
f.getFlux(starID).head()

`getLightCurve()`: If the photometry has been activated (as we ensured in this tutorial), then one can get a light curve of a specific star ID. The photometry module return the parameter `flux_type` which can be either the `estimated` or `input` flux. Let's extract the light curve for the first star:

In [ ]:
df = f.getLightCurve(starID[0], flux_type="estimated")
df.head()

`plotLightCurve()`: We can also easily visualise the light curve with:

In [ ]:
f.plotLightCurve(starID[0]);

`getMaskUpdateEvents()`: We can fetch exposure for when the mask has been updated using:

In [ ]:
updates = f.getMaskUpdateEvents()
updates

The photometric aperture mask used to extract the light curve can be fetched with the function `getApertureMask()`. We note that this has to be done individually per star. Here we use the mask updates events to get the mask used:

In [ ]:
row, col, update, npix, NSR = f.getApertureMask(starID=starID[0], imageNr=updates)
row, col, update, npix, NSR

To get the pixels of the photometric mask. exposureNr is when the mask was created, and afterwards reused for imageNr. Note that if imageNr is not given as argument then this function returns all mask update events present from the output file:

In [ ]:
#          EXAMPLE:                                                                                                                                                                                           
#                 >>> file = SimFile("Simul01.hdf5")                                                                                                                                                          
#                 >>> starID = 15563                                                                                                                                                                          
#                 >>> time, row, col, xFP, yFP = file.getPositionTimeSeries(starID)  

### 5.1.3 - Weighted photometry

We assume that each pixel value $f_{ij}$ is composed of a signal $s_{ij}$, a background $b_{ij}$, and noise $\epsilon_{ij}$:  

$$ f_{ij} = s_{ij} + b_{ij} + \varepsilon_{ij}$$

The background $b_{ij}$ is assumed to be known (in practice: read from the HDF5 file), and subtracted first. The total flux of a star is then computed using a weighted sum:

$$ F \equiv \sum\limits_{ij} \alpha \ w_{ij} (f_{ij} - b_{ij}) $$

where the weights are exponentially decaying:

$$w_{ij} = \exp\left(-\frac{(i-i_0)^2 + (j-j_0)^2}{2\sigma^2_{\rm PSF}}\right)$$

The exact (fractional) pixel coordinates $(i_0, j_0)$ of the star will be known using the Gaia catalog. For now, they are assumed to be known (i.e. read from the HDF5 file) as well.

#### Figure out the width of the PSF

In [ ]:
# stop
outputFile = h5py.File("photometrySimulation.hdf5")

In [ ]:
psf = array(outputFile["/PSF/rebinnedPSFsubPixel"])
NsubPixels = outputFile["/InputParameters/SubField"].attrs["SubPixels"]
Npixels = int(psf.shape[0]/NsubPixels)
print("PSF consists of {0}x{0} pixels, with each pixel made up of {1}x{1} subpixels".format(Npixels, NsubPixels))

Plot the PSF to see what we're dealing with.

In [ ]:
plt.pcolormesh(psf, cmap=cm.nipy_spectral, vmax=0.01)                         # 'vmax' to get a prettier image
plt.grid(True, which='major', axis='both', linestyle='-', color='w')     # To get a 'pixel-grid' on top of it.
plt.xticks(arange(0, psf.shape[0], NsubPixels))
plt.yticks(arange(0, psf.shape[1], NsubPixels))
plt.xlim(0, psf.shape[0])
plt.ylim(0, psf.shape[1])

In [ ]:
sigmaPSF = phot.computePSFsigma(psf, NsubPixels, 500)
print("Standard deviation of the PSF (assuming symmetry): {0}".format(sigmaPSF))

The following function extracts the flux of all stars in every image in the HDF5 simulation output file, and writes the results to a second HDF5 file. Depending on the number of exposures, this can take a while.

In [ ]:
print(outputFile["Flatfield"])

In [ ]:
reload(phot)
print("Start")
phot.photometry("photometrySimulation.hdf5", "photometryOutput.hdf5",starIDsInImage0, maxNexposures=1000)
print("Done")

What did the photometry script do?
- Estimate the bias and subtract it
- Correct for open shutter smearing using the smearing maps
- Convert from [ADU] to [electrons] using the (FEE and CCD) gain
- Correct for the flatfield
- Subtract the background (assumed to be known)
- Do the weighted mask photometry, assuming the correct star positions

In [ ]:
%ls -lh photometryOutput.hdf5

### Plot the positions of stars brighter than V=13 on top

Find out which stars bright

In [ ]:
allStarIDs, RA, dec, Vmag, xFPmm, yFPmm, rowPix, colPix = simfile.getStarCatalog()
brightStarIDs = allStarIDs[Vmag < 13.0]
print(f"Number of bright stars: {len(brightStarIDs)}")

Get all starIDs that are on image nr. 0. Note that these IDs are not necessarily the same as 'allStarIDs'. The latter contains all stars that have been detected, even if only in a few images because the star jittered in the subfield.

In [ ]:
starIDsInImage0, row, col, Xmm, Ymm, flux = simfile.getStarCoordinates(0)

Get the array indices of the bright stars in the 'starIDsInImage0' array. This requires a numpy trick.

In [ ]:
brightStarIndices = np.arange(len(starIDsInImage0))[np.in1d(starIDsInImage0, brightStarIDs)]

Now we can extract the row and column values of the bright stars:

In [ ]:
rowBrightStars = row[brightStarIndices]
colBrightStars = col[brightStarIndices]

Over-plot these row and column coordinates 

In [ ]:
plt.pcolormesh(image, cmap=cm.jet)
plt.scatter(floor(colBrightStars)+0.5, floor(rowBrightStars)+0.5, marker='x', c='lightgreen')
plt.xlim(0,100)
plt.ylim(0,100)

---
## 5.2 - Photometry of a variable source
---

In [ ]:
sim["ObservingParameters/NumExposures"]           = 100
sim["Sky/IncludeVariableSources"]                 = "no"
sim["Sky/VariableSourceList"]                     = "myVariableStar.txt"
sim["Sky/IncludeCosmicsInSubField"]               = "yes"
sim["Sky/IncludeCosmicsInSmearingMap"]            = "yes"
sim["Sky/IncludeCosmicsInBiasMap"]                = "yes"   
sim["PSF/Model"]                                  = "AnalyticNonGaussian"
sim["Photometry/IncludePhotometry"]               = "yes"

### What does a photometry output HDF5 file look like?

In [ ]:
photFile = h5py.File("photometryOutput.hdf5")

In [ ]:
for group in photFile["/"]:
    print(group)

In [ ]:
for group in photFile["/Photometry"]:
    print(group)

In [ ]:
for dataset in photFile["/Photometry/Exposure000000"]:
    print(dataset)

Get the photometry done on the very first image. Fluxes are expressed in [electrons/exposure]

In [ ]:
starID           = array(photFile["/Photometry/Exposure000000/starID"])
inputFlux        = array(photFile["/Photometry/Exposure000000/trueFlux"])
estimatedFlux    = array(photFile["/Photometry/Exposure000000/estimatedFlux"])
varEstimatedFlux = array(photFile["/Photometry/Exposure000000/varFlux"])
Vmag             = array(photFile["/Photometry/Exposure000000/Vmag"])
NSR              = array(photFile["/Photometry/Exposure000000/NSR"])
maskSize         = array(photFile["/Photometry/Exposure000000/maskSize"]) 

The theoretical photon noise limit is not saved in HDF5 but is easily computable using the 'inputFlux'. If we sort the photon noise value, we can plot them in one smooth line. Note the 'inputFlux' stored in the HDF5 file is an electron flux, already taking into account the transmission efficiency, and the QE. 

In [ ]:
photonLimit = inputFlux/sqrt(inputFlux)
sorted = argsort(photonLimit)[::-1]

Plot the Signal-to-Noise ratio (per exposure) of all stars in the subfield, together with theoretical photon noise limit.

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.scatter(Vmag, 1.0 / NSR)
# plt.scatter(Vmag, SNR)
plt.plot(Vmag[sorted], photonLimit[sorted], c="r")
plt.grid(True, which='major', axis='both', linestyle='-', color='gray')
plt.xlabel("V magnitude")
plt.ylabel("S/N")
# plt.xlim(7,15)
# plt.ylim(0,1600)

The large S/N values at the faint end are caused by nearby brighter stars that significantly contaminate the flux level. Even for lower magnitudes, the S/N ratio is well below the photon noise limit because of the weighted photometry gives a lower weight to the tails of the PSF. You can of course play with the PSF-width you feed to phot.photometr(), to verify if it gives better results. For the very brightest stars, the electrons flowing away due to blooming are not recovered when extracting the stellar flux.

## Extract the time series of one particular star

We choose one particular 11th magnitude star

In [ ]:
print(Vmag)
print(starID)
plt.plot(starID, Vmag, 'k+')
print(Vmag[Vmag==11.0])

In [ ]:
myStarID = 15553#9789

In [ ]:
time, trueRow, trueCol, trueFlux, outputFlux = phot.getPhotometryTimeSeries("photometryOutput.hdf5", myStarID)

In [ ]:
flux = outputFlux

Convert the absolute flux [electrons/exposure] into a relative flux [ppm]

In [ ]:
relativeFlux = (flux - flux.mean())/flux.mean()*1.e6 

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, flux)
plt.xlabel("Time")
plt.ylabel("Flux [e-/exp]")

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, relativeFlux)
plt.xlabel("Time")
plt.ylabel("Relative flux [ppm]")

## Generate stellar granulation

In [ ]:
time =  array(outputFile["/StarPositions/Time"])

In [ ]:
timeScale = array([2060.0])       # sec
varScale  = array([520.0])        # ppm

More decent values for Sun-like stars can be obtained from Pallé, P.L., Roca-Cortés, T., & Jimenez, A. 1999, ASP
Conf. Ser. 173, Stellar Structure: Theory and Test of Convective Energy Transport, ed. A. Gimenez, E.F. Guinan, &
B. Montesinos, 297.

In [ ]:
granulation = noise.redNoise(time, timeScale, varScale) 

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, granulation)
plt.xlabel("Time [s]")
plt.ylabel("Signal [ppm]")

## Generate an exoplanet signature

In [ ]:
import transits

In [ ]:
t0 = 5 * 3600.
flatPartDuration = 3 * 3600.
transitDuration  = 4 * 3600.
orbitalPeriod    = 36 * 3600.
relativeDepth    = 0.001         # 1000 ppm
phase, signal, deltaMagnitude = transits.simpleTransit(time, t0, flatPartDuration, transitDuration, orbitalPeriod, relativeDepth)

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, signal)
plt.ylim(0.997, 1.002)
plt.ticklabel_format(style='plain', axis='y', useOffset=False)

## Combine everything

In [ ]:
flux *= signal
relativeFlux = (flux - flux.mean())/flux.mean()*1.e6     # ppm
relativeFlux += granulation

Bin the time series to make the structure inside more clear

In [ ]:
binnedMean, binEdges, binNumber = binned_statistic(time, relativeFlux, statistic='mean', bins=100)
binCenters = (binEdges[1:] + binEdges[:-1])/2.0

Now plot the original time series as well as the binned time series.

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.plot(time, relativeFlux, c='b')
plt.plot(binCenters, binnedMean, c='w', linewidth=3)
plt.xlabel("Time [s]")
plt.ylabel("Relative Flux [ppm]")

## Plot the power spectral density of the time series

In [ ]:
def FFTpowerdensity(signal, timestep):

    fourier = fft.rfft(signal)
    Ntime = len(signal)
    Nfreq = len(fourier)
  
    powerdensity = abs(fourier)**2 / Ntime * timestep
    freq = arange(float(Nfreq)) / (Nfreq-1) * 0.5 / timestep
  
    return (freq, powerdensity)

In [ ]:
freq, psd = FFTpowerdensity(relativeFlux, (time[1]-time[0])*1.e-6)

The 1.e-6 is to convert the time unit from [s] to [Ms], so that the frequency is in [microHz].

In [ ]:
fig = plt.figure(figsize = (10, 10))
plt.loglog(freq, psd)
plt.xlabel("Frequency [microHz]")
plt.ylabel("Power Spectral Density [ppm^2/microHz]")